# Sentiment Analysis using IMDb Top 1000 Dataset
(Custom Sentiment from Ratings)

In [23]:
import numpy as np
import pandas as pd
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Load Dataset

In [36]:
df = pd.read_csv("imdb_top_1000.csv")
df = df[['Overview', 'IMDB_Rating']].dropna()
# Create labels
threshold = df['IMDB_Rating'].median()
df['sentiment'] = df['IMDB_Rating'].apply(
    lambda x: 1 if x >= threshold else 0
)
df.head()

,Overview,IMDB_Rating,sentiment
0,Two imprisoned men bond over a number of years...,9.3,1
1,An organized crime dynasty's aging patriarch t...,9.2,1
2,When the menace known as the Joker wreaks havo...,9.0,1
3,The early life and career of Vito Corleone in ...,9.0,1
4,A jury holdout attempts to prevent a miscarria...,9.0,1


# Data Understanding

In [37]:
print("Dataset Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())
print("\nSample Text:\n", df['Overview'].iloc[0])

Dataset Shape: (1000, 3)

Class Distribution:
 sentiment
1    569
0    431
Name: count, dtype: int64

Sample Text:
 Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.


## NLP Preprocessing

In [38]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Tokenization
    tokens = text.split()
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

In [39]:
df['clean_text'] = df['Overview'].apply(preprocess_text)

## Feature Engineering

In [40]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df['clean_text'])
y = df['sentiment']

## Train Test Split

In [41]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

## Model Training

In [44]:
def train_models(X_train, y_train):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=300),
        "Naive Bayes": MultinomialNB(),
        "Decision Tree": DecisionTreeClassifier()
    }
    
    trained_models = {}
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        trained_models[name] = model
    
    return trained_models

## Evaluation

In [45]:
def evaluate_models(models, X_test, y_test, method_name):
    print(f"\n--- {method_name} ---")
    
    for name, model in models.items():
        y_pred = model.predict(X_test)
        
        print(f"\n{name}")
        print("Accuracy:", accuracy_score(y_test, y_pred))
        print("Precision:", precision_score(y_test, y_pred, zero_division=0))
        print("Recall:", recall_score(y_test, y_pred, zero_division=0))
        print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))

In [46]:
bow_models = train_models(X_train_bow, y_train)
evaluate_models(bow_models, X_test_bow, y_test, "Bag of Words")


--- Bag of Words ---

Logistic Regression
Accuracy: 0.525
Precision: 0.556390977443609
Recall: 0.6727272727272727
F1 Score: 0.6090534979423868

Naive Bayes
Accuracy: 0.52
Precision: 0.5573770491803278
Recall: 0.6181818181818182
F1 Score: 0.5862068965517241

Decision Tree
Accuracy: 0.59
Precision: 0.6014492753623188
Recall: 0.7545454545454545
F1 Score: 0.6693548387096774


In [47]:
tfidf_models = train_models(X_train_tfidf, y_train)
evaluate_models(tfidf_models, X_test_tfidf, y_test, "TF-IDF")


--- TF-IDF ---

Logistic Regression
Accuracy: 0.555
Precision: 0.5538461538461539
Recall: 0.9818181818181818
F1 Score: 0.7081967213114754

Naive Bayes
Accuracy: 0.525
Precision: 0.5418994413407822
Recall: 0.8818181818181818
F1 Score: 0.671280276816609

Decision Tree
Accuracy: 0.52
Precision: 0.5472972972972973
Recall: 0.7363636363636363
F1 Score: 0.627906976744186


## Conclusion
- Sentiment created from ratings
- TF-IDF used for features
- Logistic Regression performs best generally